In [3]:
"""
Steg 1: Utforska JobTech JobSearch API
Kör:  pip install requests pandas
      python explore_jobtech.py
"""

import json
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests

BASE_URL = "https://jobsearch.api.jobtechdev.se/search"
API_KEY = None  # Fyll i här om API:et svarar 401/403 (nyckel registreras på apirequest.jobtechdev.se)

SEARCH_TERM = "dataanalytiker"
LIMIT = 20

OUT_DIR = Path("raw")  # "bronze-tänk": spara rådata orörd innan vi gör något med den
OUT_DIR.mkdir(exist_ok=True)


def fetch(q: str, limit: int = 20, offset: int = 0) -> dict:
    headers = {"accept": "application/json"}
    if API_KEY:
        headers["api-key"] = API_KEY
    params = {"q": q, "limit": limit, "offset": offset}
    r = requests.get(BASE_URL, params=params, headers=headers, timeout=30)
    print(f"HTTP {r.status_code} -> {r.url}")
    r.raise_for_status()
    return r.json()


def list_paths(obj, prefix="", out=None):
    """Plattar ut alla nyckel-sökvägar i en (nästlad) JSON, så vi ser vilka fält som finns."""
    if out is None:
        out = {}
    if isinstance(obj, dict):
        for k, v in obj.items():
            list_paths(v, f"{prefix}.{k}" if prefix else k, out)
    elif isinstance(obj, list):
        out[prefix + "[]"] = f"lista, {len(obj)} element"
        if obj:
            list_paths(obj[0], prefix + "[0]", out)
    else:
        out[prefix] = repr(obj)[:70]
    return out


if __name__ == "__main__":
    data = fetch(SEARCH_TERM, limit=LIMIT)

    # 1. Spara rådata med tidsstämpel
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    raw_file = OUT_DIR / f"jobsearch_{SEARCH_TERM}_{ts}.json"
    raw_file.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"\nRådata sparad: {raw_file}")

    # 2. Översikt på toppnivå
    print("\nToppnivå-nycklar:", list(data.keys()))
    print("Totalt antal träffar:", data.get("total"))
    hits = data.get("hits", [])
    print("Antal hämtade annonser:", len(hits))

    if not hits:
        raise SystemExit("Inga träffar — prova ett annat sökord.")

    # 3. Alla fält i första annonsen
    print("\n--- Fältstruktur i första annonsen ---")
    for path, example in list_paths(hits[0]).items():
        print(f"{path:60} {example}")

    # 4. Snabb tabell över några intressanta fält
    df = pd.json_normalize(hits)
    wanted = [c for c in [
        "id", "headline", "publication_date", "employer.name",
        "workplace_address.region", "workplace_address.municipality",
        "occupation.label", "employment_type.label",
    ] if c in df.columns]
    print("\n--- Översikt ---")
    print(df[wanted].to_string(max_colwidth=40))

    df.to_csv(OUT_DIR / f"jobsearch_{SEARCH_TERM}_{ts}_flat.csv", index=False)
    print(f"\nTotalt {df.shape[1]} kolumner efter utplattning.")

HTTP 200 -> https://jobsearch.api.jobtechdev.se/search?q=dataanalytiker&limit=20&offset=0

Rådata sparad: raw/jobsearch_dataanalytiker_20260921_125253.json

Toppnivå-nycklar: ['total', 'positions', 'query_time_in_millis', 'result_time_in_millis', 'stats', 'freetext_concepts', 'hits']
Totalt antal träffar: {'value': 26}
Antal hämtade annonser: 20

--- Fältstruktur i första annonsen ---
relevance                                                    1.0
id                                                           '31500847'
external_id                                                  None
original_id                                                  None
label[]                                                      lista, 0 element
webpage_url                                                  'https://arbetsformedlingen.se/platsbanken/annonser/31500847'
logo_url                                                     'https://arbetsformedlingen.se/rest/employer-logo-api/api/v1/organisat
headline  

In [4]:
import collections

for term in ["data engineer", "data scientist", "dataanalytiker", "BI-utvecklare"]:
    d = fetch(term, limit=100)
    hits = d["hits"]
    print(f"\n=== {term}: {d['total']['value']} träffar, tolkat som {d['freetext_concepts']['occupation']}")
    print(collections.Counter(h["occupation"]["label"] for h in hits).most_common(6))
    for h in hits[:5]:
        print("  -", h["headline"][:70])

HTTP 200 -> https://jobsearch.api.jobtechdev.se/search?q=data+engineer&limit=100&offset=0

=== data engineer: 81 träffar, tolkat som ['data engineer']
[('Dataingenjör', 43), ('Systemutvecklare/Programmerare', 7), ('Data scientist', 5), ('Data Warehouse specialist', 5), ('Databasutvecklare', 4), ('Mjukvaruutvecklare', 4)]
  - Data Engineer
  - Data Engineer
  - Data Engineer
  - Data Engineer
  - Data Engineer
HTTP 200 -> https://jobsearch.api.jobtechdev.se/search?q=data+scientist&limit=100&offset=0

=== data scientist: 35 träffar, tolkat som ['data scientist']
[('Data scientist', 29), ('Statistiker', 2), ('Systemutvecklare/Programmerare', 1), ('Privatekonomisk rådgivare', 1), ('Applikationskonsult', 1), ('Mjukvaruutvecklare', 1)]
  - Data Scientist
  - Data Scientist
  - Data Scientist
  - Data Scientist
  - Senior Data Scientist
HTTP 200 -> https://jobsearch.api.jobtechdev.se/search?q=dataanalytiker&limit=100&offset=0

=== dataanalytiker: 26 träffar, tolkat som ['dataanalytiker']
[('D

In [6]:
"""
Steg 2: Utforska Jobtechs historiska annons-API
Kör i samma Colab-session (behöver requests, collections).
"""

import collections
import requests

HIST = "https://historical.api.jobtechdev.se"

# 1. Fråga API:et vilka parametrar /search tar (API-specen)
spec = None
for path in ["/swagger.json", "/openapi.json", "/v1/swagger.json"]:
    r = requests.get(HIST + path, timeout=30)
    print(f"{path}: HTTP {r.status_code}")
    if r.ok and "paths" in r.text:
        spec = r.json()
        break

if spec:
    print("\nEndpoints:", list(spec["paths"].keys()))
    search = next((v for k, v in spec["paths"].items() if k.rstrip("/").endswith("search")), None)
    if search:
        params = search.get("get", {}).get("parameters", [])
        print("\nParametrar för search:")
        for p in params:
            print(f"  {p.get('name'):30} {(p.get('description') or '')[:80]}")

# 2. Provsökningar: hur många data engineer-annonser finns historiskt, och per år?
def hist_search(**params):
    r = requests.get(HIST + "/search", params=params, timeout=60)
    print(f"HTTP {r.status_code} -> {r.url}")
    r.raise_for_status()
    return r.json()

d = hist_search(q="data engineer", limit=100)
print("\nTotalt historiskt 'data engineer':", d.get("total"))
years = collections.Counter((h.get("publication_date") or "")[:4] for h in d.get("hits", []))
print("År bland de 100 första träffarna:", sorted(years.items()))
if d.get("hits"):
    print("Fält i en historisk annons:", sorted(d["hits"][0].keys()))
import pandas as pd
rows = []
for term in ["data engineer", "data scientist", "dataanalytiker", "BI-utvecklare"]:
    for year in range(2012, 2027):
        d = hist_search(q=term, limit=0, **{"published-after": f"{year}-01-01T00:00:00",
                                            "published-before": f"{year}-12-31T23:59:59"})
        rows.append({"term": term, "år": year, "antal": d["total"]["value"]})
print(pd.DataFrame(rows).pivot(index="år", columns="term", values="antal"))


/swagger.json: HTTP 200

Endpoints: ['/ad/{id}', '/search', '/stats']

Parametrar för search:
  x-feature-freetext-bool-method Boolean method to use for unclassified freetext words. Defaults to "or"
  x-feature-disable-smart-freetext Disables machine learning enriched queries. Freetext becomes traditional freetex
  x-feature-enable-false-negative Enables extra search for the current known term in free text to avoid false nega
  published-before               Fetch job ads published before specified date and time (valid formats: YYYY-mm-d
  published-after                Fetch job ads published after specified date and time (valid formats: YYYY-mm-dd
  occupation-name                One or more occupational codes according to the taxonomy
  occupation-group               One or more occupational group codes according to the taxonomy
  occupation-field               One or more occupational area codes according to the taxonomy
  occupation-collection          One or more occupational col